# Monte Carlo Convergence Analysis

Run batched simulations, plot champion-probability stability vs iteration count, then set `N_SIMULATIONS_DEFAULT` in `src/config.py`.

Prerequisite: trained models in `models/` (`python train_models.py`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import load_and_preprocess
from src.models import load_models
from src.simulations import run_monte_carlo_checkpoints
from src.convergence import build_convergence_history, estimate_stable_n, plot_convergence
from src.config import CONVERGENCE_BATCH_SIZE, CONVERGENCE_MAX_ITERATIONS, CONVERGENCE_TOL, CONVERGENCE_MIN_N, N_SIMULATIONS_DEFAULT
print('Project root:', ROOT)

In [ ]:
DATA_DIR = ROOT / 'data'
MODEL_DIR = ROOT / 'models'
OUT_DIR = ROOT / 'notebooks'
OUT_DIR.mkdir(exist_ok=True)

_, team_hist, elo_dict = load_and_preprocess(str(DATA_DIR))
models = load_models(str(MODEL_DIR))
print('Data and models loaded.')

In [ ]:
checkpoints = run_monte_carlo_checkpoints(
    team_hist=team_hist,
    elo_df=elo_dict,
    models=models,
    batch_size=CONVERGENCE_BATCH_SIZE,
    max_iterations=CONVERGENCE_MAX_ITERATIONS,
    stop_on_convergence=True,
)
print('Checkpoints recorded:', len(checkpoints))
print('Final N:', checkpoints[-1]['n_simulations'])

In [ ]:
wide, long = build_convergence_history(checkpoints)
wide.to_csv(OUT_DIR / 'convergence_history_wide.csv', index=False)
long.to_csv(OUT_DIR / 'convergence_history_long.csv', index=False)
wide.tail(10)

In [ ]:
plot_convergence(wide, long, output_path=OUT_DIR / 'convergence_plot.png', tol=CONVERGENCE_TOL)

In [ ]:
recommended = estimate_stable_n(wide, min_n=CONVERGENCE_MIN_N)
final_n = checkpoints[-1]['n_simulations']

print('Current N_SIMULATIONS_DEFAULT:', N_SIMULATIONS_DEFAULT)
print('Sweep final N:', final_n)
if recommended:
    print('Recommended stable N (advisory):', recommended)
    print('-> Update src/config.py: N_SIMULATIONS_DEFAULT =', recommended)
else:
    print('No stable N found under current tol/min_n. Inspect plot or raise CONVERGENCE_MAX_ITERATIONS.')